In [1]:
!pip install langchain-groq


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_groq import ChatGroq

In [3]:
!pip install langchain
!pip install langchain-google-genai


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [5]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

print(GOOGLE_API_KEY)

In [12]:

"""User Clarification and Research Brief Generation.

This module implements the scoping phase of the research workflow, where we:
1. Assess if the user's request needs clarification
2. Generate a detailed research brief from the conversation

The workflow uses structured output to make deterministic decisions about
whether sufficient context exists to proceed with research.
"""
from datetime import datetime
from typing_extensions import Literal

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage, get_buffer_string
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from deepresearch.state import AgentState, ClarifyWithUser, ResearchQuestion, AgentInputState
from deepresearch.prompt import clarify_with_user_instructions, transform_messages_into_research_topic_prompt
from deepresearch.utils import get_today_str
from langchain_google_genai import ChatGoogleGenerativeAI



In [14]:

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0
)

#return type meaning the node must return to only those 2 mentioned literal nodes

In [15]:
def clarify_with_user(state: AgentState) -> Command[Literal["write_research_brief"]]:
    """
    Determine if the user's request contains sufficient information to proceed with research.

    Uses structured output to make deterministic decisions and avoid hallucination.
    Routes to either research brief generation or ends with a clarification question.
    """

    # Set up structured output model
    structured_output_model = model.with_structured_output(ClarifyWithUser)

    # Invoke the model with clarification instructions
    response = structured_output_model.invoke([
        HumanMessage(
            content=clarify_with_user_instructions.format(
                messages=get_buffer_string(messages=state["messages"]),
                date=get_today_str()
            )
        )
    ])

    # Route based on clarification need
    if response.need_clarification:
        return Command(
            goto=END,
            update={"messages": [AIMessage(content=response.question)]}
        )
    else:
        return Command(
            goto="write_research_brief",
            update={"messages": [AIMessage(content=response.verification)]}
        )

In [16]:
def write_research_brief(state: AgentState):
    """
    Transform the conversation history into a comprehensive research brief.

    Uses structured output to ensure the brief follows the required format
    and contains all necessary details for effective research.
    """

    # Set up structured output model
    structured_output_model = model.with_structured_output(ResearchQuestion)

    # Generate research brief from conversation history
    response = structured_output_model.invoke([
        HumanMessage(
            content=transform_messages_into_research_topic_prompt.format(
                messages=get_buffer_string(state.get("messages", [])),
                date=get_today_str()
            )
        )
    ])

    # Update state with generated research brief and pass it to the supervisor
    return {
        "research_brief": response.research_brief,
        "supervisor_messages": [
            HumanMessage(content=f"{response.research_brief}")
        ]
    }

In [17]:
deep_research_builder=StateGraph(AgentState,input=AgentInputState)

C:\Users\Shebin\AppData\Local\Temp\ipykernel_29004\3941757231.py:1: LangGraphDeprecatedSinceV05: `input` is deprecated and will be removed. Please use `input_schema` instead. Deprecated in LangGraph V0.5 to be removed in V2.0.
  deep_research_builder=StateGraph(AgentState,input=AgentInputState)


In [18]:
deep_research_builder.add_node("clarify_with_user", clarify_with_user)
deep_research_builder.add_node("write_research_brief", write_research_brief)

In [19]:
deep_research_builder.add_edge(START, "clarify_with_user")
deep_research_builder.add_edge("write_research_brief", END)

In [20]:
from langgraph.checkpoint.memory import InMemorySaver

In [21]:
memory=InMemorySaver()
scope=deep_research_builder.compile(checkpointer=memory)

In [22]:
from langchain_core.messages import HumanMessage
config={"configurable":{"thread_id":"1"}}
result=scope.invoke({"messages":[HumanMessage(content="I want to know the best IPL team")]},config=config)

In [23]:
result['messages']

[HumanMessage(content='I want to know the best IPL team', additional_kwargs={}, response_metadata={}, id='ef4b559d-483d-4eb2-9531-b6dc9a04e913'),
 AIMessage(content='What criteria would you like to use to determine the best IPL team (e.g. recent performance, overall wins, fan base, etc.)?', additional_kwargs={}, response_metadata={}, id='079dd2c8-a49e-47c0-aa56-8e0ce30294f1', tool_calls=[], invalid_tool_calls=[])]

In [24]:
result=scope.invoke({"messages":[HumanMessage(content="I want to know the best IPL team based on no of trophies")]},config=config)

In [26]:
result['research_brief']

'I want to know the best Indian Premier League (IPL) team based on the number of trophies won, considering all seasons played so far, without any specific time constraint or recent performance bias, and using official IPL sources or reputable sports websites for accurate and up-to-date information.'

In [10]:


model = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GOOGLE_API_KEY")
)

response = model.invoke("hello")

print(response.content)

Hello. How can I assist you today?
